# Search and Download NISAR GSLC Data

This notebook demonstrates how to:
1. Search for NISAR GSLC data using the CMR API
2. Download selected granules using nisar_db.download
3. Get the North America shape polygon for filtering

In [1]:
from nisar_db.download import download_earthdata_granule

/u/aurora-r0/govorcin/miniconda/miniforge/envs/nisar-db-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NISAR_DB_GRANULE_ID = "G3817504902-ASF"
a = download_earthdata_granule(NISAR_DB_GRANULE_ID)[0]

In [16]:
import geopandas as gpd
gdf = gpd.read_file(a)
gdf

,track,frame,satelliteLat,satelliteLon,satelliteHeight,passDirection,velocityAlongTrack,velocityVertical,startET,endET,...,produceSMST,radioFrequencyInterference,insarPSFCriteria,insarProcessor,offsetsProcessor,mapTopLeftX,mapTopLeftY,mapBottomRightX,mapBottomRightY,geometry
0,1,1,-0.000021,-0.220159,753246.798651,Ascending,7571.358283,-8.720640,7.259977e+08,7.259977e+08,...,False,default,solid_earth,cpu,None,85680.0,10042560.0,421200.0,9714960.0,"MULTIPOLYGON (((-3.89189 -1.91248, -4.5857 -2...."
1,1,2,1.999994,-0.653654,752977.232273,Ascending,7571.632668,-7.213011,7.259977e+08,7.259978e+08,...,False,default,solid_earth,cpu,None,37440.0,263520.0,372960.0,-64800.0,"MULTIPOLYGON (((-4.32159 0.08322, -5.01431 -0...."
2,1,3,4.000008,-1.087872,752752.047315,Ascending,7571.848349,-5.702108,7.259978e+08,7.259978e+08,...,False,default,solid_earth,cpu,None,-10800.0,483840.0,324720.0,155520.0,"MULTIPOLYGON (((-4.75649 2.07853, -5.44897 1.9..."
3,1,4,6.000022,-1.523553,752571.264110,Ascending,7572.005312,-4.195396,7.259978e+08,7.259978e+08,...,True,default,solid_earth,cpu,None,604800.0,705600.0,942480.0,375120.0,"MULTIPOLYGON (((-5.19732 4.07342, -5.89043 3.9..."
4,1,5,8.000036,-1.961454,752434.691652,Ascending,7572.103744,-2.700181,7.259978e+08,7.259979e+08,...,True,default,solid_earth,cpu,None,553680.0,925920.0,892080.0,594720.0,"MULTIPOLYGON (((-5.64488 6.06789, -6.33946 5.9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30443,173,172,-10.000092,1.962322,755244.815965,Ascending,7569.122993,-15.950670,7.270343e+08,7.270344e+08,...,False,default,solid_earth,cpu,None,318240.0,8939520.0,651600.0,8614800.0,"MULTIPOLYGON (((-1.79699 -11.89663, -2.50958 -..."
30444,173,173,-8.000077,1.521320,754761.044026,Ascending,7569.681952,-14.570003,7.270344e+08,7.270344e+08,...,False,default,solid_earth,cpu,None,273600.0,9159840.0,606960.0,8835120.0,"MULTIPOLYGON (((-2.21105 -9.89905, -2.91804 -1..."
30445,173,174,-6.000063,1.083339,754318.392713,Ascending,7570.186239,-13.149436,7.270344e+08,7.270344e+08,...,False,default,solid_earth,cpu,None,227520.0,9380880.0,561600.0,9055440.0,"MULTIPOLYGON (((-2.62691 -7.90185, -3.32925 -8..."
30446,173,175,-4.000049,0.647600,753917.928761,Ascending,7570.634428,-11.696182,7.270344e+08,7.270345e+08,...,False,default,solid_earth,cpu,None,180720.0,9601200.0,515520.0,9275040.0,"MULTIPOLYGON (((-3.04523 -5.90502, -3.74384 -6..."


In [24]:
import sys
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
import concurrent.futures
import time

In [6]:
# Add the parent directory to sys.path
sys.path.insert(0, str(Path().absolute().parent / "src"))

from nisar_db.download import download_earthdata_granule
from nisar_db.geodb import get_opera_na_shape
from nisar_db.filenames import NISARCollection

## Search for NISAR GSLC Data

Use the CMR API to search for NISAR GSLC data.

In [7]:
def search_nisar_gslc(start_date=None, end_date=None, max_results=100, provider="ASF", 
                   use_umm_json=False, page_size=2000, max_workers=4):
    """
    Search for NISAR GSLC data in EarthData CMR.
    
    Parameters
    ----------
    start_date : str, optional
        Start date in YYYY-MM-DD format
    end_date : str, optional
        End date in YYYY-MM-DD format
    max_results : int, optional
        Maximum number of results to return
    provider : str, optional
        Provider name, default is "ASF"
    use_umm_json : bool, optional
        If True, use UMM-JSON format which includes more metadata
    page_size : int, optional
        Number of results per page, max is 2000
    max_workers : int, optional
        Number of concurrent workers for pagination
        
    Returns
    -------
    pandas.DataFrame
        DataFrame containing granule information
    """
    # Use the correct collection name
    short_name = NISARCollection.GSLC_BETA_V1_SHORT_NAME
    
    # Select CMR API endpoint based on format
    if use_umm_json:
        cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.umm_json"
    else:
        cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"
    
    # Build query parameters
    params = {
        "provider": provider,
        "short_name": short_name,
        "page_size": min(page_size, 2000),  # CMR limit is 2000
        "sort_key": "-start_date",  # Most recent first
    }
    
    # Add temporal parameters
    if start_date and end_date:
        params["temporal"] = f"{start_date}T00:00:00Z,{end_date}T23:59:59Z"
    elif start_date:
        params["temporal"] = f"{start_date}T00:00:00Z,"
    elif end_date:
        params["temporal"] = f",{end_date}T23:59:59Z"
        
    print(f"Searching for NISAR GSLC data with parameters: {params}")
    
    # First request to get total hits
    first_params = params.copy()
    first_params["page_num"] = 1
    response = requests.get(cmr_url, params=first_params, timeout=60)
    response.raise_for_status()
    
    # Get total hits from CMR header
    total_hits = int(response.headers.get("CMR-Hits", "0"))
    page_size = params.get("page_size", 2000)
    total_pages = (total_hits + page_size - 1) // page_size
    
    print(f"CMR has {total_hits} total hits across {total_pages} pages")
    
    # Function to fetch a specific page
    def fetch_page(page_num):
        page_params = params.copy()
        page_params["page_num"] = page_num
        try:
            # Add a small delay to avoid overwhelming the server
            if page_num > 1:
                time.sleep(0.2 * (page_num % max_workers))
                
            print(f"Fetching page {page_num}/{total_pages}")
            response = requests.get(cmr_url, params=page_params, timeout=60)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Error fetching page {page_num}: {e}")
            return None
    
    # Get first page results (already fetched)
    first_page_results = response.json()
    all_results = [first_page_results]
    
    # Fetch remaining pages in parallel
    if total_pages > 1:
        # Calculate actual number of workers - don't use more workers than pages
        actual_workers = min(max_workers, total_pages - 1)
        print(f"Fetching remaining {total_pages - 1} pages using {actual_workers} workers")
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=actual_workers) as executor:
            # Submit all page fetches (starting from page 2)
            future_to_page = {
                executor.submit(fetch_page, page_num): page_num
                for page_num in range(2, total_pages + 1)
            }
            
            # Collect results as they complete
            completed = 0
            for future in concurrent.futures.as_completed(future_to_page):
                page_results = future.result()
                if page_results:
                    all_results.append(page_results)
                
                completed += 1
                if completed % 5 == 0 or completed == total_pages - 1:
                    print(f"Progress: {completed}/{total_pages - 1} pages")
    
    # Extract entries from all results
    entries = []
    for result in all_results:
        if use_umm_json:
            if "items" in result:
                entries.extend(result["items"])
        else:
            if "feed" in result and "entry" in result["feed"]:
                entries.extend(result["feed"]["entry"])
    
    print(f"Found {len(entries)} GSLC products")
    
    # Extract granule information based on format
    granule_info = []
    
    if use_umm_json:
        # Process UMM-JSON format
        for item in entries[:max_results]:
            granule_id = item.get("id", "")
            umm = item.get("umm", {})
            title = umm.get("title", "")
            
            # Get temporal information
            time_start = ""
            time_end = ""
            if "temporalExtent" in umm and "rangeDateTime" in umm["temporalExtent"]:
                range_dt = umm["temporalExtent"]["rangeDateTime"]
                time_start = range_dt.get("beginningDateTime", "")
                time_end = range_dt.get("endingDateTime", "")
            
            # Get data URL
            url = ""
            if "RelatedUrls" in umm:
                for related_url in umm["RelatedUrls"]:
                    if related_url.get("Type") == "GET DATA":
                        url = related_url.get("URL", "")
                        break
            
            # Get track, frame, direction from AdditionalAttributes
            track = None
            frame = None
            direction = None
            if "AdditionalAttributes" in umm:
                for attr in umm["AdditionalAttributes"]:
                    name = attr.get("Name", "")
                    if name == "TRACK_NUMBER" and "Values" in attr:
                        try:
                            track = int(attr["Values"][0])
                        except:
                            pass
                    elif name == "FRAME_NUMBER" and "Values" in attr:
                        try:
                            frame = int(attr["Values"][0])
                        except:
                            pass
                    elif name == "ASCENDING_DESCENDING" and "Values" in attr:
                        direction = "A" if attr["Values"][0] == "ASCENDING" else "D"
            
            granule_info.append({
                "granule_id": granule_id,
                "title": title,
                "time_start": time_start,
                "time_end": time_end,
                "track": track,
                "frame": frame,
                "direction": direction,
                "url": url,
                "umm_json": umm,  # Include full UMM metadata
            })
    else:
        # Process standard JSON format
        for entry in entries[:max_results]:
            granule_id = entry.get("id", "")
            title = entry.get("title", "")
            time_start = entry.get("time_start", "")
            time_end = entry.get("time_end", "")
            
            # Get URL
            url = ""
            if "links" in entry:
                for link in entry["links"]:
                    if "rel" in link and "data" in link["rel"]:
                        url = link.get("href", "")
                        break
            
            # Extract track, frame, etc from title
            # Example: NISAR_L2_PR_GSLC_010_164_D_076_2005_QPDH_A_20260120T140558_20260120T140633_X05010_N_F_J_001
            parts = title.split("_")
            
            track = None
            frame = None
            direction = None
            
            try:
                # Find the position of "GSLC" marker to anchor our parsing
                gslc_pos = -1
                for i, part in enumerate(parts):
                    if part == "GSLC" or part == "PR_GSLC":
                        gslc_pos = i
                        break
                
                # If found, parse based on relative positions
                if gslc_pos >= 0 and gslc_pos + 3 < len(parts):
                    try:
                        track = int(parts[gslc_pos + 1])
                    except:
                        pass
                    
                    try:
                        direction = parts[gslc_pos + 3]
                    except:
                        pass
                    
                    try:
                        frame = int(parts[gslc_pos + 4])
                    except:
                        pass
            except:
                pass
            
            granule_info.append({
                "granule_id": granule_id,
                "title": title,
                "time_start": time_start,
                "time_end": time_end,
                "track": track,
                "frame": frame,
                "direction": direction,
                "url": url,
            })
    
    return pd.DataFrame(granule_info)

In [8]:
# Define search parameters
today = datetime.now()
one_year_ago = today - timedelta(days=365)
start_date = one_year_ago.strftime("%Y-%m-%d")
end_date = today.strftime("%Y-%m-%d")

# Set the correct collection name
short_name = "NISAR_L2_GSLC_BETA_V1"

# Search for NISAR GSLC data
print(f"Searching for NISAR GSLC data with collection: {short_name}")
cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"

# Build query parameters
params = {
    "provider": "ASF",
    "short_name": short_name,
    "page_size": 20,
    "sort_key": "-start_date",  # Most recent first
}

# Add temporal parameters
if start_date and end_date:
    params["temporal"] = f"{start_date}T00:00:00Z,{end_date}T23:59:59Z"
    
print(f"Search parameters: {params}")

# Make the request
response = requests.get(cmr_url, params=params, timeout=60)
response.raise_for_status()

# Parse the response
results = response.json()

# Extract entries
entries = []
if "feed" in results and "entry" in results["feed"]:
    entries = results["feed"]["entry"]
    print(f"Found {len(entries)} GSLC products")
else:
    print("No results found")

# Convert to DataFrame
granule_info = []
for entry in entries:
    granule_id = entry.get("id", "")
    title = entry.get("title", "")
    time_start = entry.get("time_start", "")
    time_end = entry.get("time_end", "")
    
    # Get URL
    url = ""
    if "links" in entry:
        for link in entry["links"]:
            if "rel" in link and "data" in link["rel"]:
                url = link.get("href", "")
                break
    
    granule_info.append({
        "granule_id": granule_id,
        "title": title,
        "time_start": time_start,
        "time_end": time_end,
        "url": url
    })

gslc_df = pd.DataFrame(granule_info)

# Display results
if not gslc_df.empty:
    display(gslc_df[["title", "url"]].head())
else:
    print("No results to display")

Searching for NISAR GSLC data with collection: NISAR_L2_GSLC_BETA_V1
Search parameters: {'provider': 'ASF', 'short_name': 'NISAR_L2_GSLC_BETA_V1', 'page_size': 20, 'sort_key': '-start_date', 'temporal': '2025-04-22T00:00:00Z,2026-04-22T23:59:59Z'}
Found 20 GSLC products


,title,url
0,NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_202...,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
1,NISAR_L2_PR_GSLC_010_164_D_077_2005_QPDH_A_202...,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
2,NISAR_L2_PR_GSLC_010_164_D_076_2005_QPDH_A_202...,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
3,NISAR_L2_PR_GSLC_010_164_D_075_2005_QPDH_A_202...,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
4,NISAR_L2_PR_GSLC_010_164_D_074_2005_QPDH_A_202...,https://nisar.asf.earthdatacloud.nasa.gov/NISA...


In [9]:
# Function to explore specific UMM-JSON fields
def explore_umm_field(df, field_name, max_items=5):
    """Explore a specific field in UMM-JSON metadata."""
    if 'umm_json' not in df.columns or df.empty:
        print("No UMM-JSON metadata found.")
        return
    
    print(f"\nExploring field: {field_name}")
    count = 0
    
    for i, row in df.iterrows():
        if field_name in row['umm_json']:
            print(f"\nItem {count + 1}:")
            value = row['umm_json'][field_name]
            if isinstance(value, dict):
                print(json.dumps(value, indent=2))
            elif isinstance(value, list):
                print(json.dumps(value[:3], indent=2))  # Show first 3 items
                if len(value) > 3:
                    print(f"... and {len(value) - 3} more items")
            else:
                print(value)
                
            count += 1
            if count >= max_items:
                break
    
    if count == 0:
        print(f"No items found with field '{field_name}'")

# Example usage: explore RelatedUrls field
if 'gslc_umm_df' in locals() and not gslc_umm_df.empty:
    explore_umm_field(gslc_umm_df, "RelatedUrls", 2)
else:
    print("No UMM-JSON data available to explore.")

No UMM-JSON data available to explore.


In [10]:
# Extract all S3 and HTTPS paths for bulk access
if 'gslc_umm_df' in locals() and not gslc_umm_df.empty:
    # Extract paths to CSV files
    if 's3_url' in gslc_umm_df.columns and 'https_url' in gslc_umm_df.columns:
        # Save URLs to CSV
        urls_df = gslc_umm_df[['title', 'https_url', 's3_url']]
        urls_df.to_csv("nisar_gslc_urls.csv", index=False)
        print(f"Saved {len(urls_df)} GSLC URLs to nisar_gslc_urls.csv")
        
        # Also create separate files for S3 and HTTPS URLs
        with open("nisar_gslc_s3_urls.txt", "w") as f:
            for s3_url in gslc_umm_df['s3_url'].dropna():
                if s3_url:
                    f.write(f"{s3_url}\n")
        
        with open("nisar_gslc_https_urls.txt", "w") as f:
            for https_url in gslc_umm_df['https_url'].dropna():
                if https_url:
                    f.write(f"{https_url}\n")
                    
        print(f"Saved S3 URLs to nisar_gslc_s3_urls.txt")
        print(f"Saved HTTPS URLs to nisar_gslc_https_urls.txt")
        
        # Print example usage with aws s3 cp
        s3_urls = [url for url in gslc_umm_df['s3_url'].dropna() if url]
        if s3_urls:
            print("\nExample usage with AWS CLI:")
            print(f"aws s3 cp {s3_urls[0]} ./")
else:
    print("No URLs available to extract.")

No URLs available to extract.


In [11]:
# Now search using UMM-JSON format to get additional metadata
print("\nSearching for NISAR GSLC data using UMM-JSON format...")
cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.umm_json"

# Use the same parameters but for UMM-JSON format
response = requests.get(cmr_url, params=params, timeout=60)
response.raise_for_status()

# Parse the response
results = response.json()

# Extract items
umm_items = []
if "items" in results:
    umm_items = results["items"]
    print(f"Found {len(umm_items)} items with UMM-JSON format")
else:
    print("No UMM-JSON items found")

# Convert to DataFrame with UMM metadata
umm_granule_info = []
for item in umm_items:
    granule_id = item.get("id", "")
    umm = item.get("umm", {})
    title = umm.get("title", "")
    
    # Get temporal information
    time_start = ""
    time_end = ""
    if "temporalExtent" in umm and "rangeDateTime" in umm["temporalExtent"]:
        range_dt = umm["temporalExtent"]["rangeDateTime"]
        time_start = range_dt.get("beginningDateTime", "")
        time_end = range_dt.get("endingDateTime", "")
    
    # Get data URLs (both HTTPS and S3)
    https_url = ""
    s3_url = ""
    if "RelatedUrls" in umm:
        for related_url in umm["RelatedUrls"]:
            if related_url.get("Type") == "GET DATA":
                if related_url.get("Subtype") == "AMAZON S3":
                    s3_url = related_url.get("URL", "")
                elif "https:" in related_url.get("URL", ""):
                    https_url = related_url.get("URL", "")
    
    umm_granule_info.append({
        "granule_id": granule_id,
        "title": title,
        "time_start": time_start,
        "time_end": time_end,
        "https_url": https_url,
        "s3_url": s3_url,
        "umm_json": umm  # Store the full UMM metadata
    })

gslc_umm_df = pd.DataFrame(umm_granule_info)

# Display results
if not gslc_umm_df.empty:
    display(gslc_umm_df[["title", "https_url", "s3_url"]].head())
    
    # Show sample UMM metadata structure
    if len(gslc_umm_df) > 0 and "umm_json" in gslc_umm_df.columns:
        umm_sample = gslc_umm_df.iloc[0]["umm_json"]
        print("\nSample UMM metadata structure (keys only):")
        print(", ".join(sorted(umm_sample.keys())))
else:
    print("No UMM-JSON results to display")


Searching for NISAR GSLC data using UMM-JSON format...
Found 20 items with UMM-JSON format


,title,https_url,s3_url
0,,https://nisar.asf.earthdatacloud.nasa.gov/NISA...,
1,,https://nisar.asf.earthdatacloud.nasa.gov/NISA...,
2,,https://nisar.asf.earthdatacloud.nasa.gov/NISA...,
3,,https://nisar.asf.earthdatacloud.nasa.gov/NISA...,
4,,https://nisar.asf.earthdatacloud.nasa.gov/NISA...,



Sample UMM metadata structure (keys only):
AdditionalAttributes, CollectionReference, DataGranule, GranuleUR, InputGranules, MetadataSpecification, OrbitCalculatedSpatialDomains, PGEVersionClass, Platforms, ProviderDates, RelatedUrls, SpatialExtent, TemporalExtent


In [12]:
# Now search using UMM-JSON format to get additional metadata
print("\nSearching for NISAR GSLC data using UMM-JSON format...")
gslc_umm_df = search_nisar_gslc(
    start_date=start_date,
    end_date=end_date,
    max_results=20,
    use_umm_json=True
)

# Display results
if not gslc_umm_df.empty:
    display(gslc_umm_df[['title', 'track', 'frame', 'direction', 'url']].head())
    
    # Show the full UMM metadata for the first product
    print("\nSample UMM metadata for first product:")
    if 'umm_json' in gslc_umm_df.columns and len(gslc_umm_df) > 0:
        umm_sample = gslc_umm_df.iloc[0]['umm_json']
        print(json.dumps(umm_sample, indent=2)[:1000] + "...\n(truncated)")
else:
    print("No results found with UMM-JSON format.")


Searching for NISAR GSLC data using UMM-JSON format...
Searching for NISAR GSLC data with parameters: {'provider': 'ASF', 'short_name': 'NISAR_L2_GSLC_BETA_V1', 'page_size': 2000, 'sort_key': '-start_date', 'temporal': '2025-04-22T00:00:00Z,2026-04-22T23:59:59Z'}
CMR has 23450 total hits across 12 pages
Fetching remaining 11 pages using 4 workers
Fetching page 4/12
Fetching page 5/12
Fetching page 2/12
Fetching page 3/12
Fetching page 6/12
Fetching page 7/12
Fetching page 8/12
Fetching page 9/12
Progress: 5/11 pages
Fetching page 10/12
Fetching page 12/12
Fetching page 11/12
Progress: 10/11 pages
Progress: 11/11 pages
Found 23450 GSLC products


,title,track,frame,direction,url
0,,165,100,D,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
1,,164,77,D,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
2,,164,76,D,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
3,,164,75,D,https://nisar.asf.earthdatacloud.nasa.gov/NISA...
4,,164,74,D,https://nisar.asf.earthdatacloud.nasa.gov/NISA...



Sample UMM metadata for first product:
{
  "TemporalExtent": {
    "RangeDateTime": {
      "BeginningDateTime": "2026-01-20T15:59:30.000000Z",
      "EndingDateTime": "2026-01-20T15:59:50.999342Z"
    }
  },
  "OrbitCalculatedSpatialDomains": [
    {
      "OrbitNumber": 2513
    }
  ],
  "GranuleUR": "NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001",
  "AdditionalAttributes": [
    {
      "Name": "ASCENDING_DESCENDING",
      "Values": [
        "DESCENDING"
      ]
    },
    {
      "Name": "FRAME_NUMBER",
      "Values": [
        "100"
      ]
    },
    {
      "Name": "FREQUENCIES",
      "Values": [
        "A",
        "B"
      ]
    },
    {
      "Name": "FREQUENCY_A_POLARIZATION",
      "Values": [
        "HH",
        "HV"
      ]
    },
    {
      "Name": "FREQUENCY_A_POLARIZATION_CONCAT",
      "Values": [
        "HH+HV"
      ]
    },
    {
      "Name": "FREQUENCY_A_RANGE_BANDWIDTH",
      "Values": [
        "20"
    

In [13]:
# Create a download directory
download_dir = Path("./downloads")
download_dir.mkdir(exist_ok=True)

# Select the first granule to download
if not gslc_df.empty:
    selected_granule = gslc_df.iloc[0]
    print(f"Downloading granule: {selected_granule['title']}")
    
    # Download the granule
    files = download_earthdata_granule(
        selected_granule["granule_id"], 
        output_dir=download_dir, 
        skip_existing=True
    )
    
    print(f"Downloaded files: {files}")
else:
    print("No granules to download")

# You can also directly download using the URL
if not gslc_df.empty and 'url' in gslc_df.columns and gslc_df.iloc[0]['url']:
    print("\nAlternatively, you can download directly using the URL:")
    url = gslc_df.iloc[0]['url']
    print(f"URL: {url}")
    print("Command to download:")
    print(f"curl -L -c cookies.txt -b cookies.txt -n \"{url}\" -o download.h5")

Downloaded files: ['downloads/NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001.h5', 'downloads', 'downloads/granules']

Alternatively, you can download directly using the URL:
URL: https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_BETA_V1/NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001/NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001.h5
Command to download:
curl -L -c cookies.txt -b cookies.txt -n "https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GSLC_BETA_V1/NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001/NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001.h5" -o download.h5


In [14]:
# Create a download directory
download_dir = Path("./downloads")
download_dir.mkdir(exist_ok=True)

# Select the first granule to download
if not gslc_df.empty:
    selected_granule = gslc_df.iloc[0]
    print(f"Downloading granule: {selected_granule['title']}")
    
    # Download the granule
    files = download_earthdata_granule(
        selected_granule["granule_id"], 
        output_dir=download_dir, 
        skip_existing=True
    )
    
    print(f"Downloaded files: {files}")
else:
    print("No granules to download")

Downloaded files: ['downloads/NISAR_L2_PR_GSLC_010_165_D_100_2005_DHDH_M_20260120T155930_20260120T155950_X05010_N_P_J_001.h5', 'downloads', 'downloads/granules']


## Get North America Shape

Retrieve the OPERA North America shape for filtering.

In [17]:
# Get North America shape
na_shape = get_opera_na_shape()

# Print basic information about the shape
print(f"North America shape type: {na_shape.geom_type}")
print(f"North America shape area: {na_shape.area:.2f}")
print(f"North America shape bounds: {na_shape.bounds}")

NameError: name 'get_opera_na_shape' is not defined

In [ ]:
# Plot the North America shape
import geopandas as gpd
from shapely.geometry import mapping

# Convert to GeoDataFrame for plotting
na_gdf = gpd.GeoDataFrame({"geometry": [na_shape]}, crs="EPSG:4326")

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
na_gdf.plot(ax=ax, alpha=0.5)
ax.set_title("OPERA North America Region")
plt.tight_layout()
plt.show()

In [ ]:
# Create a list of GSLC file paths
if not gslc_df.empty:
    with open("gslc_granules.csv", "w") as f:
        gslc_df.to_csv(f, index=False)
    
    with open("gslc_files.txt", "w") as f:
        for _, row in gslc_df.iterrows():
            f.write(f"{row['title']}\n")
    
    print(f"Saved {len(gslc_df)} GSLC granules to gslc_granules.csv")
    print(f"Saved {len(gslc_df)} GSLC file paths to gslc_files.txt")
else:
    print("No GSLC files to save")

## Next Steps

Now you can use the GSLC data with other nisar_db tools:

1. Create a GSLC catalog: `nisar-db create-catalog --input gslc_files.txt --output gslc_catalog.csv`
2. Create a consistent GSLC catalog: `nisar-db create-consistent --input gslc_catalog.csv --output consistent_gslc_catalog.json`
3. Create a frame-to-bound JSON: `nisar-db create-frame-to-bound --nisar-gpkg NISAR_TrackFrame_L_YYYYMMDD.gpkg --output nisar-frame-to-bounds.json`

## Getting All NISAR GSLC Products

There are approximately 23,450 GSLC products available in the CMR system. To retrieve all of them efficiently, you can use the provided `get_all_nisar_gslc.py` script:

```bash
python get_all_nisar_gslc.py --max-results 25000 --output all_gslc.csv
```

This script uses parallel workers to efficiently paginate through all available products and saves them to a CSV file.

In [ ]:
# Get all fields from UMM-JSON metadata
def get_umm_fields(umm_df):
    """Get all available fields in UMM-JSON metadata."""
    if 'umm_json' not in umm_df.columns or umm_df.empty:
        print("No UMM-JSON metadata found.")
        return set()
    
    # Collect all top-level fields
    fields = set()
    for i, row in umm_df.iterrows():
        if isinstance(row['umm_json'], dict):
            fields.update(row['umm_json'].keys())
    
    return fields

# Display available fields in UMM-JSON metadata
if 'gslc_umm_df' in locals() and not gslc_umm_df.empty:
    fields = get_umm_fields(gslc_umm_df)
    print("Available UMM-JSON fields:")
    print(", ".join(sorted(fields)))
else:
    print("No UMM-JSON data available.")

In [ ]:
# Function to extract S3 paths from UMM-JSON
def extract_s3_paths(umm_df):
    """Extract S3 paths from UMM-JSON metadata."""
    s3_paths = []
    
    if 'umm_json' not in umm_df.columns:
        print("No UMM-JSON metadata found.")
        return pd.DataFrame()
    
    for i, row in umm_df.iterrows():
        umm = row['umm_json']
        title = row['title']
        
        # Find S3 URL
        s3_url = None
        if "RelatedUrls" in umm:
            for url_info in umm["RelatedUrls"]:
                # Look for URLs with "AMAZON S3" or "S3" in the subtype or description
                if ("Description" in url_info and "S3" in url_info["Description"]) or \
                   ("Subtype" in url_info and "S3" in url_info["Subtype"]):
                    s3_url = url_info.get("URL", None)
                    break
        
        s3_paths.append({
            'title': title,
            's3_path': s3_url,
            'https_url': row.get('url', None)
        })
    
    return pd.DataFrame(s3_paths)

# Extract S3 paths if we have UMM-JSON data
if 'gslc_umm_df' in locals() and not gslc_umm_df.empty:
    s3_paths_df = extract_s3_paths(gslc_umm_df)
    display(s3_paths_df.head())
else:
    print("No UMM-JSON data available to extract S3 paths.")

## Extract S3 Paths from UMM-JSON

If you need to access the data via S3 instead of HTTPS, you can extract the S3 paths from the UMM-JSON metadata.

## Next Steps

Now you can use the list of GSLC files with other nisar_db tools:

1. Create a GSLC catalog: `nisar-db create-catalog --input gslc_files.txt --output gslc_catalog.csv`
2. Create a consistent GSLC catalog: `nisar-db create-consistent --input gslc_catalog.csv --output consistent_gslc_catalog.json`
3. Create a frame-to-bound JSON: `nisar-db create-frame-to-bound --nisar-gpkg NISAR_TrackFrame_L_YYYYMMDD.gpkg --output nisar-frame-to-bounds.json`